In [ ]:
import gzip
import pandas as pd
import numpy as np
import scipy.sparse as sp
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

# =====================================================================
# 1. FONCTIONS UTILITAIRES
# =====================================================================
def is_timestamp(val):
    return isinstance(val, str) and val.startswith('t') and val[1:].isdigit()

def get_seq_duration(seq):
    timestamps = [int(x[1:]) for x in seq if is_timestamp(x)]
    return max(timestamps) if timestamps else 0

def get_macro_actions(seq):
    return [x.split('(')[0] for x in seq]

def charger_donnees(chemin_fichier, has_target=True):
    chemin = Path(chemin_fichier)
    
    is_gzipped = False
    try:
        with gzip.open(chemin, 'rb') as f:
            f.read(1)
        is_gzipped = True
    except OSError: 
        pass

    open_fn = gzip.open if is_gzipped else open
    records = []
    
    # Compteur pour générer le RowId de 1 à N exigé par Kaggle
    row_id_counter = 1 
    
    with open_fn(chemin, "rt", encoding="utf-8") as f:
        for line in f:
            elements = [e.strip() for e in line.strip().split(",") if e.strip()]
            if elements:
                if has_target:
                    # Dans train.csv : [user_id, navigateur, actions...]
                    records.append({
                        'user_id': elements[0], 
                        'navigator': elements[1], 
                        'sequence': elements[2:]
                    })
                else:
                    # Dans test.csv : [navigateur, actions...] (L'ID a disparu !)
                    records.append({
                        'RowId': row_id_counter, 
                        'navigator': elements[0], 
                        'sequence': elements[1:]
                    })
                    row_id_counter += 1
    
    index_col = 'user_id' if has_target else 'RowId'
    return pd.DataFrame(records).set_index(index_col)

def ingenieur_features(df):
    df['actions'] = df['sequence'].apply(lambda liste: [x for x in liste if not is_timestamp(x)])
    df['macro_actions'] = df['actions'].apply(get_macro_actions)
    df['seq_duration'] = df['sequence'].apply(get_seq_duration)
    df['seq_len'] = df['actions'].apply(len)
    df['actions_per_sec'] = (df['seq_len'] / df['seq_duration'].replace(0, np.nan)).fillna(0)
    
    df['phrase_actions'] = df['macro_actions'].apply(
        lambda liste: " ".join([act.replace(" ", "_") for act in liste])
    )
    return df

# =====================================================================
# 2. PRÉPARATION DU JEU D'ENTRAÎNEMENT (100% DES DONNÉES)
# =====================================================================
print("1/5 - Chargement et préparation des données d'entraînement...")
df_train = charger_donnees("train.csv", has_target=True) # ou "train.csv" selon ton dossier
df_train = ingenieur_features(df_train)

# =====================================================================
# 3. VECTORISATION & ENTRAÎNEMENT DU MODÈLE FINAL
# =====================================================================
print("2/5 - Vectorisation TF-IDF et Scaling...")
vectoriseur = TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 3), max_features=3000)
X_train_nlp = vectoriseur.fit_transform(df_train['phrase_actions'])

scaler = StandardScaler()
X_train_num = scaler.fit_transform(df_train[['seq_duration', 'actions_per_sec']].fillna(0))

df_train_nav = pd.get_dummies(df_train['navigator'], prefix='nav')
colonnes_navigateurs = df_train_nav.columns 

X_train_final = sp.hstack([X_train_nlp, X_train_num, df_train_nav.values])
y_train_final = df_train.index

print("3/5 - Entraînement du LinearSVC sur l'intégralité du Train Set...")
modele_final = LinearSVC(C=1.0, class_weight='balanced', max_iter=5000, random_state=42)
modele_final.fit(X_train_final, y_train_final)

# =====================================================================
# 4. PRÉPARATION DU JEU DE TEST KAGGLE
# =====================================================================
print("4/5 - Chargement et transformation du Test Set Kaggle...")
df_test = charger_donnees("test.csv", has_target=False)
df_test = ingenieur_features(df_test)

X_test_nlp = vectoriseur.transform(df_test['phrase_actions'])
X_test_num = scaler.transform(df_test[['seq_duration', 'actions_per_sec']].fillna(0))

df_test_nav = pd.get_dummies(df_test['navigator'], prefix='nav')
df_test_nav = df_test_nav.reindex(columns=colonnes_navigateurs, fill_value=0)

X_test_final = sp.hstack([X_test_nlp, X_test_num, df_test_nav.values])

# =====================================================================
# 5. PRÉDICTION & EXPORT
# =====================================================================
print("5/5 - Génération des prédictions...")
predictions = modele_final.predict(X_test_final)

submission = pd.DataFrame({
    'RowId': df_test.index,
    'prediction': predictions
})

fichier_sortie = "submission.csv"
submission.to_csv(fichier_sortie, index=False)

print(f"\n✅ Terminé ! Le fichier '{fichier_sortie}' est généré et son format est corrigé.")

1/5 - Chargement et préparation des données d'entraînement...
2/5 - Vectorisation TF-IDF et Scaling...
3/5 - Entraînement du LinearSVC sur l'intégralité du Train Set...
4/5 - Chargement et transformation du Test Set Kaggle...
5/5 - Génération des prédictions...

✅ Terminé ! Le fichier 'submission.csv' est généré et son format est corrigé.
